In [1]:
import numpy as np
import pandas as pd



In [2]:
train_path = "/kaggle/input/hms-harmful-brain-activity-classification/train.csv"
train_df = pd.read_csv(train_path)

vote_columns = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]
# Global class probabilities (unused in the current baseline but retained)
class_votes = train_df[vote_columns].sum().astype(float)
class_probs = (class_votes / class_votes.sum()).values.astype(float)



In [3]:
test_path = "/kaggle/input/hms-harmful-brain-activity-classification/test.csv"
test_df = pd.read_csv(test_path)



In [4]:
col_names = ["eeg_id"] + vote_columns

rng = np.random.RandomState(42)  # deterministic randomness
alpha = np.full(len(vote_columns), 0.1)  # Dirichlet concentration (very sparse)
prob_matrix = rng.dirichlet(alpha, size=test_df.shape[0])

submission = pd.DataFrame(
    np.column_stack((test_df["eeg_id"].values, prob_matrix)), columns=col_names
)

# Round to 4 decimals
submission[vote_columns] = np.round(submission[vote_columns].astype(float), 4)

# Replace any rows that summed to zero after rounding with a tiny epsilon
epsilon = 1e-6
zero_rows = submission[vote_columns].sum(axis=1) == 0
if zero_rows.any():
    submission.loc[zero_rows, vote_columns] = epsilon

# Renormalise so each row sums to 1
row_sums = submission[vote_columns].sum(axis=1)
submission[vote_columns] = submission[vote_columns].div(row_sums, axis=0)



In [5]:
output_path = "/kaggle/working/submission.csv"
submission.to_csv(output_path, index=False)
print(f"Submission saved to {output_path}")

Submission saved to /kaggle/working/submission.csv
